# 4 — Is This Just Financial Feature Engineering? Testing Against Harder Baselines

Notebook 3 confirmed TDA adds value beyond plain financial features. A
sharp objection follows naturally: financial data can be engineered
dynamically too (rolling stats, EWM trends) -- maybe *any* sufficiently
rich dynamic feature set would show a similar gain, and TDA isn't special
at all. This notebook tests that directly, then keeps raising the bar:
first against engineered financial dynamics, then against GARCH -- the
actual industry-standard volatility model.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

DATA = Path.cwd().parent / "data" / "processed" / "backtest_results"


def paired_bootstrap_test(y_true, score_a, score_b, metric_fn, n_bootstrap=1000, random_state=42):
    """Two-sided paired bootstrap test for metric_fn(y_true, score_b) -
    metric_fn(y_true, score_a). Resamples row indices (with replacement)
    jointly across y_true/score_a/score_b, preserving the pairing.

    Inlined here (unchanged) from this project's own tested implementation
    (regime_detection/src/backtest/stats.py) so this notebook has no
    dependency on the rest of that codebase."""
    y_true = np.asarray(y_true)
    score_a = np.asarray(score_a)
    score_b = np.asarray(score_b)
    n = len(y_true)

    observed_a = metric_fn(y_true, score_a)
    observed_b = metric_fn(y_true, score_b)
    observed_diff = observed_b - observed_a

    rng = np.random.default_rng(random_state)
    diffs = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        try:
            diff = metric_fn(y_true[idx], score_b[idx]) - metric_fn(y_true[idx], score_a[idx])
        except ValueError:
            continue
        if np.isfinite(diff):
            diffs.append(diff)
    diffs = np.array(diffs)

    prop_le_0 = np.mean(diffs <= 0)
    prop_ge_0 = np.mean(diffs >= 0)
    p_value = float(min(1.0, 2 * min(prop_le_0, prop_ge_0)))
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5]) if len(diffs) else (np.nan, np.nan)

    return {
        "metric_a": float(observed_a),
        "metric_b": float(observed_b),
        "observed_diff": float(observed_diff),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "p_value": p_value,
        "n_bootstrap_valid": len(diffs),
    }


## Step 1: The fairness check -- financial dynamics alone

The exact same dynamic-transform toolkit used to build the TDA features
(rolling stats, EWM, autocorrelation) was applied to Arm A's own 26
financial fields -- zero topology, zero blockchain data -- producing
**Arm J**.


In [2]:
arm_j = pd.read_parquet(DATA / "model_comparison_results_arm_j.parquet")
row = arm_j[(arm_j["target"] == "vol_regime_h7") & (arm_j["model"] == "random_forest")].iloc[0]
print(f"Arm A (plain financial):            AUC {row['metric_a']:.4f}")
print(f"Arm J (+ financial dynamics):        AUC {row['metric_b']:.4f}  (gain: +{row['observed_diff']:.4f}, FDR p={row['p_value_fdr']:.4f})")
print()
print("This is the single largest individual-arm effect found anywhere in this project --")
print("bigger than any of the four TDA families in notebook 3, using zero TDA data.")


Arm A (plain financial):            AUC 0.4796
Arm J (+ financial dynamics):        AUC 0.5979  (gain: +0.1183, FDR p=0.0000)

This is the single largest individual-arm effect found anywhere in this project --
bigger than any of the four TDA families in notebook 3, using zero TDA data.


**The real question, then**: does TDA still add anything *on top of*
this much stronger baseline -- not just on top of plain Arm A?


In [3]:
gen_rows = []
k = pd.read_parquet(DATA / "model_comparison_results_arm_k.parquet")
gen_rows.append(("Arm K (Stage A dynamics beyond Arm J)", k[(k["target"] == "vol_regime_h7") & (k["model"] == "random_forest")].iloc[0]))
lmn = pd.read_parquet(DATA / "model_comparison_results_arm_lmn.parquet")
for arm, name in [("arm_l_vs_arm_j", "Arm L (trajectory beyond Arm J)"), ("arm_m_vs_arm_j", "Arm M (curvature+autocorr beyond Arm J)"), ("arm_n_vs_arm_j", "Arm N (exhaustive sweep beyond Arm J)")]:
    sub = lmn[(lmn["comparison"] == arm) & (lmn["target"] == "vol_regime_h7") & (lmn["model"] == "random_forest")]
    gen_rows.append((name, sub.iloc[0]))
o = pd.read_parquet(DATA / "model_comparison_results_arm_o.parquet")
gen_rows.append(("Arm O (D+E+G combined, the TDA ceiling)", o[(o["target"] == "vol_regime_h7") & (o["model"] == "random_forest")].iloc[0]))

for name, row in gen_rows:
    sig = "significant" if row["significant_fdr"] else "not significant"
    print(f"{name:45s} gain={row['observed_diff']:+.4f}  FDR p={row['p_value_fdr']:.4f}  ({sig})")


Arm K (Stage A dynamics beyond Arm J)         gain=+0.0254  FDR p=0.0000  (significant)
Arm L (trajectory beyond Arm J)               gain=+0.0151  FDR p=0.0000  (significant)
Arm M (curvature+autocorr beyond Arm J)       gain=+0.0132  FDR p=0.0400  (significant)
Arm N (exhaustive sweep beyond Arm J)         gain=-0.0039  FDR p=0.9920  (not significant)
Arm O (D+E+G combined, the TDA ceiling)       gain=+0.0347  FDR p=0.0000  (significant)


**3 of 4 TDA shortlists still add a significant, independently-confirmed
improvement on top of financial dynamics** (the fourth, Arm N, does not --
itself informative: its own descriptors happened to overlap more with
what financial dynamics already captures). Combining the three
confirmed-useful families (Arm O) reaches this project's TDA-only
ceiling: **RandomForest AUC 0.6325**.

## Step 2: The hardest baseline -- GARCH

Neither Arm A nor Arm J is the actual professional standard for
volatility forecasting. That's **GARCH**: a three-parameter model fit
directly on returns, capturing volatility clustering with zero feature
engineering. A plain, untuned GARCH(1,1) was walk-forward validated the
same way as everything else (verified directly: perturbing a return and
rerunning the identical construction leaves every earlier date's forecast
unchanged -- no lookahead).


In [4]:
garch = pd.read_parquet(DATA / "model_comparison_results_garch.parquet")
print(garch[["garch_model", "ml_arm", "metric_a", "metric_b", "observed_diff", "p_value_fdr", "significant_fdr"]]
      .rename(columns={"metric_a": "GARCH AUC", "metric_b": "ML arm AUC", "observed_diff": "ML - GARCH"})
      .to_string(index=False))


garch_model ml_arm  GARCH AUC  ML arm AUC  ML - GARCH  p_value_fdr  significant_fdr
      garch  arm_a   0.629002    0.479605   -0.149397        0.000             True
      garch  arm_j   0.629002    0.597868   -0.031133        0.096            False
      garch  arm_k   0.629002    0.623222   -0.005779        0.846            False
  gjr_garch  arm_a   0.626352    0.479605   -0.146747        0.000             True
  gjr_garch  arm_j   0.626352    0.597868   -0.028484        0.108            False
  gjr_garch  arm_k   0.626352    0.623222   -0.003130        0.846            False


**GARCH alone (AUC 0.629) significantly beats plain Arm A** -- but ties
Arm K and Arm O, this project's TDA-augmented ceiling (p=0.85 / 0.78).
This was the project's most important calibration moment: a three-
parameter model with zero feature engineering matched an entire feature-
engineering research arc. Not a retraction of Step 1's findings -- those
comparisons remain valid on their own terms -- but a real reframing of
what "TDA adds value" should be read to mean in practice.

## Step 3: Give GARCH's forecast to the model, then ask again

Instead of treating GARCH as an outside competitor, feed its own
conditional-volatility forecast to the model **as an input feature**,
then ask whether TDA still adds anything from that much harder starting
point. Three arms: **Arm P** (Arm A + GARCH features), **Arm Q** (+
financial dynamics), **Arm R** (+ TDA).


In [5]:
pqr = pd.read_parquet(DATA / "model_comparison_results_arm_pqr.parquet")
for comparison, label in [("arm_q_vs_arm_p", "Financial dynamics beyond GARCH (Arm Q vs. Arm P)"),
                          ("arm_r_vs_arm_q", "TDA beyond GARCH + financial dynamics (Arm R vs. Arm Q)")]:
    sub = pqr[(pqr["comparison"] == comparison) & (pqr["target"] == "vol_regime_h7") & (pqr["model"] == "random_forest")]
    row = sub.iloc[0]
    sig = "SIGNIFICANT" if row["significant_fdr"] else "not significant"
    print(f"{label}: gain={row['observed_diff']:+.4f}, FDR p={row['p_value_fdr']:.4f}  ({sig})")


Financial dynamics beyond GARCH (Arm Q vs. Arm P): gain=+0.0192, FDR p=0.5500  (not significant)
TDA beyond GARCH + financial dynamics (Arm R vs. Arm Q): gain=+0.0449, FDR p=0.0000  (SIGNIFICANT)


Financial dynamics' own advantage nearly vanishes once GARCH is already
present -- it was substantially rediscovering the same volatility-
clustering structure GARCH already encodes. **TDA does not** -- it still
adds a significant, independently-confirmed improvement even at this,
the hardest bar tested anywhere in this project, reaching **RandomForest
AUC 0.6531, the highest point estimate in the whole project.**

Does the complete stack finally beat plain GARCH outright?


In [6]:
oof_r = pd.read_parquet(DATA / "oof_predictions_arm_pqr.parquet")
garch_oof = pd.read_parquet(DATA / "oof_predictions_garch.parquet").set_index("date")

sub = oof_r[(oof_r["target"] == "vol_regime_h7") & (oof_r["model"] == "random_forest") & (oof_r["seed"] == 42) & (oof_r["arm"] == "arm_r")].set_index("date").sort_index()
common = sub.index.intersection(garch_oof.index)
res = paired_bootstrap_test(
    sub.loc[common, "y_true"].values.astype(int),
    garch_oof.loc[common, "score"].values,
    sub.loc[common, "proba_1"].values,
    roc_auc_score, n_bootstrap=5000, random_state=0,
)
print(f"Arm R (full stack) vs. GARCH alone: GARCH AUC={res['metric_a']:.4f}, Arm R AUC={res['metric_b']:.4f}, "
      f"diff={res['observed_diff']:+.4f}, p={res['p_value']:.4f}")
print()
print("Closer than anything else tried in this project (Arm O's own GARCH comparison above was p=0.78) --")
print("but not conventionally significant. A genuine near-miss, not a win.")


Arm R (full stack) vs. GARCH alone: GARCH AUC=0.6290, Arm R AUC=0.6531, diff=+0.0241, p=0.0812

Closer than anything else tried in this project (Arm O's own GARCH comparison above was p=0.78) --
but not conventionally significant. A genuine near-miss, not a win.


## Conclusion

TDA's predictive value for volatility forecasting survives every
baseline this project could construct: plain financial features,
aggressively engineered financial dynamics, and a combined baseline that
already includes GARCH. Whether the resulting full pipeline is worth
building *instead of* GARCH alone -- rather than *alongside* it -- remains
a genuinely open, near-miss question (p ~ 0.06-0.08, stable under
re-estimation with more bootstrap precision and unmoved by a
mechanism-motivated model change tried in the full project -- see
`STATUS.md`). The next notebook asks the question this all exists for:
does any of this translate into money.
